# Lab: K-Nearest Neighbors (KNN)

## 1. Ý tưởng

KNN là một trong những thuật toán đơn giản và trực quan nhất, có thể tóm tắt bằng câu "nói cho tôi biết bạn của bạn là ai, tôi sẽ nói bạn là người thế nào".

Để dự đoán nhãn của một điểm mới, KNN làm 3 bước:
1. Tính khoảng cách từ điểm mới đến mọi điểm trong tập train.
2. Lấy $k$ điểm gần nhất.
3. Bầu chọn nhãn xuất hiện nhiều nhất trong $k$ điểm đó (mode).

$$
\hat{y} = \text{mode}\{y_{(1)}, y_{(2)}, \dots, y_{(k)}\}
$$

Trong đó $y_{(i)}$ là nhãn của hàng xóm gần thứ $i$.

## 2. Đặc điểm độc đáo

- **Không có giai đoạn train**: KNN chỉ "ghi nhớ" tập train. Tất cả tính toán xảy ra lúc dự đoán, nên train nhanh còn predict chậm (phải tính khoảng cách với toàn bộ train mỗi lần).
- **Non-parametric**: không giả định phân phối dữ liệu, không có tham số học được.
- **Lazy learner**: trì hoãn mọi tính toán đến lúc cần.

### Minh hoạ: KNN ra quyết định như thế nào

![KNN với k=1 và k=5 cho hai kết quả khác nhau](images/01_cach_knn_hoat_dong.png)

*Cùng một điểm cần dự đoán (ngôi sao cam). Với $k=1$, hàng xóm gần nhất tình cờ là một điểm đỏ lạc sang vùng xanh, nên KNN trả về đỏ. Nới vòng tròn lên $k=5$, bốn điểm xanh xung quanh chiếm đa số và KNN đổi kết quả thành xanh.*

Ba điều rút ra từ hình:

1. KNN không vẽ sẵn ranh giới lúc train. Nó chỉ khoanh một vòng tròn quanh điểm cần dự đoán tại thời điểm predict rồi đếm phiếu.
2. $k$ có thể xem là bán kính tầm nhìn của mô hình. $k$ nhỏ nghĩa là mô hình chỉ dựa vào một vài điểm gần nhất, nên rất nhạy với nhiễu và outlier.
3. Cái "vòng tròn" đó là vòng tròn của khoảng cách Euclidean. Đổi metric thì hình dạng vòng này đổi, và tập hàng xóm cũng đổi theo (xem mục dưới).


## 3. Chọn k và chọn metric khoảng cách

### Hàm khoảng cách

Phổ biến nhất là **Euclidean** (L2):
$$d(u, v) = \sqrt{\sum_{i=1}^{n}(u_i - v_i)^2}$$

**Manhattan** (L1):
$$d(u, v) = \sum_{i=1}^{n}|u_i - v_i|$$

**Cosine distance** (cho dữ liệu văn bản, vector cao chiều):
$$d_{\cos}(u, v) = 1 - \frac{u \cdot v}{\|u\| \, \|v\|}$$

Lưu ý: **cosine *similarity*** là $\frac{u\cdot v}{\|u\|\|v\|}$ (càng lớn càng gần). **Cosine *distance*** là $1 - \text{similarity}$ (càng nhỏ càng gần). Đây là chỗ rất dễ nhầm: sắp xếp tăng dần cho distance, giảm dần cho similarity.

### Chọn k

- $k = 1$: nhạy nhiễu (nhìn 1 hàng xóm thì đoán theo hàng xóm đó, kể cả khi hàng xóm là outlier).
- $k$ lớn: dự đoán mượt hơn nhưng bias cao (có thể bỏ qua chi tiết).
- Chọn $k$ lẻ cho phân loại 2 lớp để tránh hoà.
- Thực tế: dùng cross-validation để tìm $k$ tối ưu, thường $k \in [3, 30]$.

## 4. Vì sao phải chuẩn hoá feature

KNN dựa trên khoảng cách, nên feature có scale lớn sẽ lấn át các feature khác. Ví dụ feature `Age` (0-100) và `Income` (0-1.000.000): chênh nhau 1 đơn vị Income lấn át chênh 1 đơn vị Age.

Quy tắc chung: luôn chuẩn hoá (`StandardScaler` hoặc `MinMaxScaler`) trước khi dùng KNN.

Một nguyên tắc quan trọng: chỉ `fit` scaler trên tập train, rồi `transform` lên test. Nếu fit trên cả tập (gồm cả test) thì thông tin của test đã lọt vào mô hình (**data leakage**) và kết quả test bị thổi phồng.

### Minh hoạ: mỗi metric có một "hình dạng" riêng

Cách dễ nhất để *nhìn thấy* một hàm khoảng cách là vẽ **quả cầu đơn vị** của nó, tức tập hợp mọi điểm cách gốc đúng 1 đơn vị. Hình dạng quả cầu này quyết định "ai được coi là hàng xóm".

![Quả cầu đơn vị của L1, L2, L-vô-cùng và Minkowski tổng quát](images/05_qua_cau_don_vi_minkowski.png)

*Trái: L1 (Manhattan) là hình thoi, L2 (Euclidean) là hình tròn, $L_\infty$ (Chebyshev) là hình vuông. Giữa: Minkowski $p$ nội suy liên tục giữa chúng: $p$ càng lớn, quả cầu càng "phình" ra thành hình vuông. Phải: với cùng một điểm truy vấn $q$, điểm A gần hơn theo L1 nhưng điểm B lại gần hơn theo L2. Đổi metric là đổi luôn kết quả dự đoán.*

Tất cả nằm trong một công thức duy nhất, **khoảng cách Minkowski bậc $p$**:

$$
d_p(u, v) = \left(\sum_{i=1}^{n} |u_i - v_i|^p\right)^{1/p}
$$

- $p=1$ cho Manhattan, $p=2$ cho Euclidean, còn $p \to \infty$ cho Chebyshev $\;d_\infty(u,v) = \max_i |u_i - v_i|$.
- Trong sklearn: `KNeighborsClassifier(metric='minkowski', p=2)` là mặc định (chính là Euclidean).
- Với $p < 1$ thì $d_p$ không còn là metric (vi phạm bất đẳng thức tam giác). Vẫn dùng được để xếp hạng hàng xóm, nhưng mất nhiều tính chất lý thuyết và không dùng được cho KD-Tree.

| Metric | Công thức | Nên dùng khi |
|---|---|---|
| Euclidean (L2) | $\sqrt{\sum (u_i-v_i)^2}$ | Mặc định cho feature số đã chuẩn hoá, số chiều thấp/vừa |
| Manhattan (L1) | $\sum \lvert u_i-v_i \rvert$ | Nhiều chiều, có outlier, hoặc feature là "số bước/số lần" rời rạc |
| Chebyshev ($L_\infty$) | $\max_i \lvert u_i-v_i \rvert$ | Chỉ quan tâm chiều lệch nhiều nhất (kiểm tra dung sai, cờ vua) |
| Cosine | $1 - \frac{u \cdot v}{\lVert u \rVert \lVert v \rVert}$ | Văn bản, TF-IDF, embedding: khi *hướng* quan trọng hơn *độ dài* |
| Hamming | tỷ lệ vị trí khác nhau | Dữ liệu nhị phân / chuỗi phân loại thuần |

Một quan sát đáng chú ý: ở số chiều cao, L1 thường phân biệt tốt hơn L2, còn L2 lại tốt hơn $L_\infty$. Lý do: $p$ càng lớn thì khoảng cách càng bị một chiều duy nhất chi phối, nên thông tin từ các chiều còn lại bị bỏ qua.


### Minh hoạ: $k$ điều khiển độ phức tạp của mô hình

![Decision boundary của KNN với k = 1, 5, 15, 50](images/02_bien_quyet_dinh_theo_k.png)

*Cùng một bộ dữ liệu "hai vầng trăng", chỉ đổi $k$. $k=1$: ranh giới lởm chởm, mỗi điểm nhiễu tự tạo một "ốc đảo" riêng, đó là overfit. $k=50$: ranh giới mượt đến mức cắt phăng cả phần đuôi của hai vầng trăng, đó là underfit. $k$ ở giữa (5 đến 15) bám đúng hình dạng thật.*

Nói cách khác, $k$ đóng vai trò của một tham số điều chuẩn (regularization). Lưu ý chiều ngược với trực giác thông thường:

$$
k \text{ nhỏ } \Longrightarrow \text{ mô hình phức tạp} \qquad\qquad k \text{ lớn } \Longrightarrow \text{ mô hình đơn giản}
$$

Ở thái cực $k = N$ (toàn bộ tập train), mọi điểm đều được dự đoán là lớp đa số và mô hình không học được gì.


### Ảnh động: ranh giới biến dạng thế nào khi $k$ tăng

![Ranh giới quyết định của KNN khi k chạy từ 1 lên 40](images/anim_k_thay_doi.gif)

*Vẫn bộ dữ liệu "hai vầng trăng" đó, chỉ có $k$ chạy từ 1 lên 40. Ở $k=1$ ranh giới bám sát từng điểm train và sinh ra hàng loạt "ốc đảo" nhỏ; $k$ càng tăng thì các ốc đảo biến mất và đường biên duỗi dần thành một đường trơn. Con số test accuracy trên mỗi khung cho thấy giá phải trả của $k=1$: chỉ 86.2%, trong khi từ $k=5$ trở đi mô hình dao động quanh 90 đến 92%. Đây là hình bốn panel ở trên, chiếu liên tục thay vì cắt lấy bốn lát.*


### Minh hoạ: chuẩn hoá đổi hẳn tập hàng xóm

![Hàng xóm gần nhất trước và sau StandardScaler](images/04_vi_sao_phai_scale.png)

*Bên trái (chưa scale): vì Income trải trên hàng trăm nghìn còn Age chỉ trải trên vài chục, khoảng cách Euclidean gần như chỉ đo chênh lệch Income. Kết quả: 5 hàng xóm của một người 35 tuổi lại là những người 32 đến 61 tuổi có cùng mức thu nhập, và biểu quyết cho kết quả sai. Bên phải (sau `StandardScaler`): hai trục có trọng lượng ngang nhau, hàng xóm là những người cùng độ tuổi và dự đoán đúng.*

Toán học đứng sau: nếu feature $j$ có độ lệch chuẩn $\sigma_j$ thì đóng góp của nó vào khoảng cách bình phương tỷ lệ với $\sigma_j^2$. Feature nào có $\sigma$ lớn gấp 1000 lần thì đóng góp lớn gấp $10^6$ lần, các feature còn lại gần như không còn ảnh hưởng.

| Bộ chuẩn hoá | Công thức | Nên dùng khi |
|---|---|---|
| `StandardScaler` | $z = \dfrac{x - \mu}{\sigma}$ | Mặc định; feature gần phân phối chuẩn |
| `MinMaxScaler` | $z = \dfrac{x - x_{\min}}{x_{\max} - x_{\min}}$ | Cần mọi feature nằm gọn trong $[0,1]$ (ví dụ có cả cột one-hot) |
| `RobustScaler` | $z = \dfrac{x - \text{median}}{IQR}$ | Dữ liệu có outlier mạnh: median/IQR không bị outlier kéo |

Lưu ý về data leakage: phải `fit` scaler chỉ trên tập train rồi `transform` lên test. Nếu `fit` trên toàn bộ dữ liệu, thông tin về $\mu, \sigma$ của tập test đã lọt vào mô hình, và điểm test sẽ cao hơn thực tế. Cách an toàn là gói vào `Pipeline`:

```python
from sklearn.pipeline import make_pipeline
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
cross_val_score(pipe, X_train, y_train, cv=5)   # scaler được fit lại trong từng fold
```

Nếu gọi `cross_val_score` trên dữ liệu đã scale sẵn bằng cả tập, ta vẫn bị leakage dù đã chia train/test.


# THỰC HÀNH 1: KNN trên Iris (4 features liên tục)

Dataset kinh điển: 150 hoa Iris, 4 đặc trưng (sepal length/width, petal length/width), 3 loài (setosa, versicolor, virginica). Mục tiêu: phân loại loài.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)

iris = pd.read_excel('data/Iris.xls')
print(f'Shape: {iris.shape}')
print(iris.head())
print(f'\nClasses: {iris.iloc[:, -1].unique()}')

In [ ]:
X = iris.iloc[:, :-1].values
y = LabelEncoder().fit_transform(iris.iloc[:, -1])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# fit scaler chỉ trên train để tránh rò rỉ dữ liệu, rồi transform test
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Train KNN với vài giá trị k khác nhau để chọn k tối ưu
ks = list(range(1, 31))
cv_scores = []
for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_s, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

best_k = ks[int(np.argmax(cv_scores))]
print(f'Best k theo 5-fold CV: {best_k}, accuracy = {max(cv_scores)*100:.2f}%')

plt.figure(figsize=(8, 4))
plt.plot(ks, [s*100 for s in cv_scores], 'o-')
plt.xlabel('k'); plt.ylabel('CV Accuracy (%)')
plt.title('Chọn k bằng cross-validation')
plt.axvline(best_k, color='red', linestyle='--', label=f'best k={best_k}')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Đánh giá trên tập test với best_k
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_s, y_train)
y_pred = knn_best.predict(X_test_s)

print(f'Test accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print()
print(classification_report(y_test, y_pred,
                            target_names=['setosa', 'versicolor', 'virginica']))

## 5. Tự cài đặt KNN từ đầu

Để hiểu KNN đang làm gì, ta tự code phiên bản từ scratch và so sánh với sklearn.

In [ ]:
from collections import Counter

class MyKNN:
    def __init__(self, k=5, metric='euclidean'):
        self.k = k
        self.metric = metric

    def fit(self, X, y):
        # KNN không có giai đoạn train thực sự, chỉ lưu lại dữ liệu.
        self.X_train = np.asarray(X)
        self.y_train = np.asarray(y)
        return self

    def _distance(self, x):
        if self.metric == 'euclidean':
            return np.sqrt(((self.X_train - x) ** 2).sum(axis=1))
        elif self.metric == 'manhattan':
            return np.abs(self.X_train - x).sum(axis=1)
        elif self.metric == 'cosine':
            num = self.X_train @ x
            den = np.linalg.norm(self.X_train, axis=1) * np.linalg.norm(x) + 1e-10
            similarity = num / den
            return 1 - similarity            # cosine distance (càng nhỏ càng gần)
        else:
            raise ValueError(f'Unknown metric: {self.metric}')

    def predict_one(self, x):
        d = self._distance(x)
        nn_idx = np.argsort(d)[:self.k]      # k khoảng cách nhỏ nhất
        labels = self.y_train[nn_idx]
        return Counter(labels).most_common(1)[0][0]

    def predict(self, X):
        return np.array([self.predict_one(np.asarray(x)) for x in X])

    def score(self, X, y):
        return (self.predict(X) == np.asarray(y)).mean()

# So sánh với sklearn
my_knn = MyKNN(k=best_k).fit(X_train_s, y_train)
acc_mine = my_knn.score(X_test_s, y_test)
acc_skl = knn_best.score(X_test_s, y_test)
print(f'My KNN  test acc: {acc_mine*100:.2f}%')
print(f'sklearn test acc: {acc_skl*100:.2f}%')

Hai con số này phải khớp nhau. Nếu lệch nhỏ thì nguyên nhân là cách phá hoà khác nhau giữa `np.argsort` và sklearn (xem mục 10).


# THỰC HÀNH 2: KNN trên drug200

Dữ liệu trộn cả feature liên tục (Age, Na_to_K) và rời rạc (Sex, BP, Cholesterol). Sau khi encode, ta có không gian số và KNN dùng được.

In [ ]:
drug = pd.read_csv('data/drug200.csv')

# One-hot cho cột rời rạc: giữ tính chất "không có thứ tự" cho Sex, Cholesterol
drug_enc = pd.get_dummies(drug, columns=['Sex', 'BP', 'Cholesterol'], drop_first=False)
X = drug_enc.drop('Drug', axis=1).values.astype(float)
y = LabelEncoder().fit_transform(drug_enc['Drug'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# scale sau khi đã split, fit chỉ trên train
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

for k in [1, 3, 5, 7, 11, 15]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    acc = knn.score(X_test_s, y_test)
    print(f'k = {k:2d}  test acc = {acc*100:.2f}%')

## 6. Phụ lục: Nearest Centroid (Rocchio)

Một họ hàng đơn giản hơn của KNN: thay vì so với mọi điểm train, ta tính **trung tâm (centroid)** của mỗi lớp rồi gán nhãn theo centroid gần nhất.

$$\mu_c = \frac{1}{|S_c|}\sum_{x \in S_c} x, \quad \hat{y}(x) = \arg\min_c \|x - \mu_c\|$$

Ưu điểm: predict O(C) thay vì O(N) như KNN. Nhược: chỉ hoạt động tốt khi mỗi lớp có dạng tròn/lồi và phương sai gần nhau.

In [ ]:
from sklearn.neighbors import NearestCentroid

# X_train_s / X_test_s / y_test ở các cell trên đã bị gán lại cho drug200,
# nên ở đây phải dựng lại tập Iris (split + scale) rồi mới so sánh. Nếu dùng
# thẳng knn_best (đã fit trên Iris 4 feature) với X_test_s (drug200, 9 feature)
# thì sklearn sẽ báo lỗi số chiều.
X_iris = iris.iloc[:, :-1].values
y_iris = LabelEncoder().fit_transform(iris.iloc[:, -1])

X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.2,
                                          random_state=42, stratify=y_iris)
scaler_iris = StandardScaler()
X_tr_s = scaler_iris.fit_transform(X_tr)
X_te_s = scaler_iris.transform(X_te)

nc = NearestCentroid()
nc.fit(X_tr_s, y_tr)

knn_iris = KNeighborsClassifier(n_neighbors=best_k)
knn_iris.fit(X_tr_s, y_tr)

print(f'Nearest Centroid trên Iris: {nc.score(X_te_s, y_te)*100:.2f}%')
print(f'KNN(k={best_k}) trên Iris : {knn_iris.score(X_te_s, y_te)*100:.2f}%')


## 7. Bias-variance của KNN theo $k$, và quy tắc $k \approx \sqrt{N}$

### 7.1. Phân rã bias-variance (bản hồi quy, dễ nhìn nhất)

Giả sử dữ liệu sinh ra bởi $y = f(x) + \varepsilon$ với $\mathbb{E}[\varepsilon]=0$, $\mathrm{Var}(\varepsilon)=\sigma^2$. Dự đoán của KNN hồi quy tại $x_0$ là trung bình $k$ hàng xóm:

$$
\hat{f}(x_0) = \frac{1}{k}\sum_{i=1}^{k} y_{(i)}
$$

Sai số kỳ vọng tách được thành ba phần:

$$
\underbrace{\mathbb{E}\big[(y_0 - \hat{f}(x_0))^2\big]}_{\text{tổng sai số}}
= \underbrace{\sigma^2}_{\text{nhiễu, không thể bỏ}}
+ \underbrace{\Big[f(x_0) - \frac{1}{k}\sum_{i=1}^{k} f(x_{(i)})\Big]^2}_{\text{bias}^2 \text{ (tăng theo } k)}
+ \underbrace{\frac{\sigma^2}{k}}_{\text{variance} \text{ (giảm theo } k)}
$$

Công thức này tóm tắt gần như toàn bộ hành vi của KNN:

- Số hạng variance là $\sigma^2/k$: tăng $k$ thì nhiễu được trung bình hoá đi, variance giảm đúng theo $1/k$.
- Số hạng bias là chênh lệch giữa $f(x_0)$ và trung bình $f$ của $k$ hàng xóm. $k$ càng lớn, hàng xóm càng phải lấy từ xa, $f$ ở đó càng khác $f(x_0)$, nên bias tăng.
- Hai số hạng đi ngược chiều nhau, nên tồn tại một $k$ tối ưu ở giữa. Hình dưới đây minh hoạ điều đó.

![Train accuracy và test accuracy của KNN theo k](images/03_bias_variance_theo_k.png)

*Trung bình trên 8 lần chia train/test để đường mượt. Ở $k=1$ train accuracy luôn bằng 100% (mỗi điểm train là hàng xóm gần nhất của chính nó) trong khi test accuracy thấp nhất: dấu hiệu kinh điển của variance cao. Khi $k$ tăng quá mức, cả hai đường cùng tụt: đó là bias. Vùng xám giữa hai đường chính là "mức overfit".*

### 7.2. Quy tắc $k \approx \sqrt{N}$ đến từ đâu?

Định lý nhất quán (Stone, 1977) nói rằng KNN hội tụ về bộ phân loại Bayes tối ưu khi $N \to \infty$ nếu đồng thời:

$$
k \to \infty \qquad \text{và} \qquad \frac{k}{N} \to 0
$$

Ý nghĩa trực giác: $k \to \infty$ để trung bình hoá hết nhiễu (variance $\to 0$), còn $k/N \to 0$ để vùng chứa $k$ hàng xóm vẫn co lại về điểm $x_0$ (bias $\to 0$). Chọn $k = \sqrt{N}$ thoả mãn cả hai: $\sqrt{N} \to \infty$ nhưng $\sqrt{N}/N = 1/\sqrt{N} \to 0$.

Tuy vậy $\sqrt{N}$ chỉ là điểm khởi đầu, vì quy tắc này bỏ qua:

- Mức nhiễu $\sigma$: dữ liệu càng nhiễu càng cần $k$ lớn hơn.
- Số chiều $d$: $d$ lớn thì $k$ hàng xóm nằm rất xa, nên cần giảm $k$ hoặc giảm chiều trước.
- Độ chồng lấn giữa các lớp và mật độ không đều: ở vùng dữ liệu thưa, $k$ cố định buộc phải với tay ra rất xa.
- Mất cân bằng lớp: $k$ lớn khiến lớp thiểu số không bao giờ thắng phiếu (xem mục 16).

Trong thực hành, lấy $k_0 = \sqrt{N}$ làm tâm, quét $k$ trong khoảng $[1, 3k_0]$ bằng `cross_val_score` (hoặc `GridSearchCV`), rồi chọn theo validation, không chọn theo test.


## 8. Hình học của 1-NN: sơ đồ Voronoi

Với $k=1$, KNN có một mô tả hình học chính xác: nó chia mặt phẳng thành các **ô Voronoi**. Ô của điểm train $x_i$ là tập mọi vị trí mà $x_i$ là điểm gần nhất:

$$
V(x_i) = \{\, z : \|z - x_i\| \le \|z - x_j\| \ \ \forall j \,\}
$$

Mỗi ràng buộc $\|z - x_i\| \le \|z - x_j\|$ là một **nửa mặt phẳng** (chia bởi đường trung trực của đoạn $x_i x_j$), nên $V(x_i)$ là giao của các nửa mặt phẳng, do đó luôn là một đa giác lồi. Ranh giới quyết định của 1-NN chính là phần biên giữa các ô có nhãn khác nhau, vì thế nó luôn là đường **gấp khúc**, ghép từ các đoạn trung trực.

![Sơ đồ Voronoi và decision boundary của 1-NN](images/08_voronoi_1nn.png)

*Trái: mỗi điểm train sở hữu một ô đa giác lồi. Phải: gộp các ô cùng nhãn lại, ta được đúng vùng quyết định của 1-NN.*

Ba hệ quả quan trọng:

1. 1-NN luôn đạt train accuracy 100% (điểm gần nhất của một điểm train là chính nó, khoảng cách 0). Vì vậy train accuracy của 1-NN không có giá trị đánh giá và không nên báo cáo.
2. Thêm hoặc bớt một điểm train là đổi ngay hình dạng ranh giới trong cả một vùng. Đó là biểu hiện trực quan của variance cao.
3. Định lý Cover-Hart (1967): khi $N \to \infty$, sai số của 1-NN thoả

$$
E^* \;\le\; E_{1\text{-NN}} \;\le\; E^*\left(2 - \frac{C}{C-1}E^*\right) \;\le\; 2E^*
$$

với $E^*$ là sai số Bayes (giới hạn lý thuyết không mô hình nào vượt qua được) và $C$ là số lớp. Nói gọn: với vô hạn dữ liệu, sai số của 1-NN không bao giờ vượt quá hai lần sai số của mô hình tốt nhất có thể. Đây là lý do KNN vẫn được dùng làm baseline nghiêm túc.


## 9. Trọng số khoảng cách: `weights='uniform'` vs `weights='distance'`

Ở dạng mặc định (`uniform`), cả $k$ hàng xóm có quyền bỏ phiếu như nhau: hàng xóm cách 0.1 và hàng xóm cách 10 có tiếng nói ngang nhau. Điều đó không hợp lý khi mật độ dữ liệu không đều.

Với `weights='distance'`, mỗi hàng xóm nhận trọng số nghịch đảo khoảng cách và điểm số của lớp $c$ là:

$$
\text{score}(c) = \sum_{i \,:\, y_{(i)} = c} w_i, \qquad w_i = \frac{1}{d_i}, \qquad
\hat{y} = \arg\max_c \ \text{score}(c)
$$

Với hồi quy thì là trung bình có trọng số: $\displaystyle \hat{y} = \frac{\sum_i w_i y_{(i)}}{\sum_i w_i}$.

![So sánh weights uniform và distance](images/07_uniform_vs_distance.png)

*Trái: 2 hàng xóm đỏ cách $q$ khoảng 0.3 đến 0.4, còn 3 hàng xóm xanh cách hơn 2.0. `uniform` đếm 3 > 2 và chọn xanh. `distance` cộng $1/d$: đỏ được $1/0.36 + 1/0.40 \approx 5.2$ phiếu còn xanh chỉ được $\approx 1.4$, nên chọn đỏ. Phải: cùng dữ liệu, `distance` cho ranh giới bám sát điểm train hơn.*

Khi nào nên dùng `weights='distance'`:

- $k$ được chọn lớn (bạn muốn ổn định) nhưng vẫn không muốn hàng xóm ở xa làm loãng kết quả.
- Mật độ dữ liệu không đều: vùng thưa buộc phải với hàng xóm rất xa.
- Có thể dùng làm cơ chế phá hoà phiếu tự nhiên (xem mục 10).

Mặt trái:

- $w_i = 1/d_i \to \infty$ khi $d_i \to 0$. sklearn xử lý riêng: nếu có điểm trùng khít ($d=0$) thì điểm đó chiếm toàn bộ trọng số. Hệ quả: train accuracy luôn bằng 100% với mọi $k$, lại một con số không dùng để đánh giá được.
- Rất nhạy với điểm trùng lặp hoặc gần trùng (dữ liệu bị nhân bản do lỗi ETL sẽ chiếm toàn bộ phiếu).
- Nếu muốn mềm hơn, dùng kernel Gaussian $w_i = \exp(-d_i^2 / 2h^2)$ thay cho $1/d$ (đó chính là ý tưởng của *kernel regression* / Nadaraya-Watson).


## 10. Hoà phiếu (tie-breaking) và vì sao chọn $k$ lẻ

KNN là thuật toán biểu quyết, mà biểu quyết thì có thể hoà. Có hai loại hoà cần phân biệt:

### 10.1. Hoà phiếu (hai lớp cùng số phiếu)

Với 2 lớp và $k$ chẵn, ví dụ $k=4$ cho 2 phiếu mỗi bên là hoà. sklearn không báo lỗi mà lặng lẽ chọn lớp có chỉ số nhãn nhỏ hơn (vì `argmax` trên vector xác suất trả về vị trí đầu tiên đạt cực đại). Nghĩa là mô hình của bạn có một thiên vị hệ thống về phía lớp `0` mà không ai nói cho bạn biết.

Vì vậy với bài toán 2 lớp, nên chọn $k$ lẻ. Khi đó $\sum_c \text{votes}_c = k$ lẻ nên không thể chia đôi.

Lưu ý rằng $k$ lẻ không loại trừ được hoà khi có từ 3 lớp trở lên. Ví dụ 3 lớp với $k=3$ hoàn toàn có thể ra 1-1-1. Với $C$ lớp, hoà là không tránh khỏi về nguyên tắc; các cách xử lý:

- `weights='distance'`: trọng số thực gần như không bao giờ bằng nhau hoàn toàn, nên tự phá hoà, và phá theo cách có ý nghĩa (ưu tiên lớp có hàng xóm gần hơn).
- Giảm $k$ xuống 1 rồi quyết định (tương đương "hàng xóm gần nhất được quyền quyết định").
- Chọn lớp có tần suất cao hơn trong tập train (thiên vị prior), chỉ nên dùng khi bạn *muốn* thiên vị đó.

### 10.2. Hoà khoảng cách (nhiều điểm cách đúng bằng nhau)

Xảy ra rất thường xuyên với dữ liệu rời rạc hoặc đã one-hot, khi nhiều điểm cách $q$ đúng bằng nhau nhưng chỉ còn 1 suất trong top-$k$. sklearn giữ điểm xuất hiện trước trong thứ tự tập train. Hệ quả thực tế: chỉ cần xáo thứ tự dòng của tập train là kết quả predict có thể đổi.

Đây cũng là lý do ở phần tự cài đặt, kết quả của `MyKNN` và sklearn có thể lệch nhau một chút: `np.argsort` và sklearn không nhất thiết phá hoà giống nhau. Nếu cần kết quả tái lập hoàn toàn, hãy cố định thứ tự dữ liệu và ghi rõ điều đó trong báo cáo.


## 11. KNN cho hồi quy với `KNeighborsRegressor`

KNN không chỉ để phân loại. Đổi "biểu quyết" thành "lấy trung bình" là ta có ngay một bộ hồi quy:

$$
\hat{y}(x_0) = \frac{1}{k}\sum_{i=1}^{k} y_{(i)}
\qquad \text{hoặc có trọng số} \qquad
\hat{y}(x_0) = \frac{\sum_{i=1}^{k} w_i\, y_{(i)}}{\sum_{i=1}^{k} w_i},\quad w_i = \frac{1}{d_i}
$$

![KNN hồi quy với k = 1, 5, 25 và weights='distance'](images/09_knn_hoi_quy_bac_thang.png)

*Với `weights='uniform'`, dự đoán là hằng số trên từng vùng, nên đường dự đoán luôn là hàm bậc thang, bậc càng to khi $k$ càng lớn. Với `weights='distance'`, đường trở nên liên tục và đi đúng qua từng điểm train.*

Ba đặc tính cần nhớ:

1. **Không ngoại suy được.** Mọi dự đoán đều là trung bình của các $y$ đã thấy, nên $\hat{y}$ luôn nằm trong $[\min y_{\text{train}}, \max y_{\text{train}}]$. Ở hai đầu miền dữ liệu, đường dự đoán bẹt ra thành đường nằm ngang (nhìn rõ ở panel $k=25$). Hồi quy tuyến tính thì ngược lại, ngoại suy thoải mái (đôi khi thoải mái quá mức).
2. **Không có công thức.** Không có hệ số nào để diễn giải, không nói được "tăng 1 đơn vị $x$ thì $y$ tăng bao nhiêu".
3. **Đánh giá bằng RMSE / MAE / $R^2$** giống mọi mô hình hồi quy khác; nhớ vẫn phải chuẩn hoá $X$ (nhưng không chuẩn hoá $y$ trừ khi bạn nhớ biến đổi ngược).


In [ ]:
# Demo độc lập: KNN cho hồi quy (chạy được mà không cần cell nào ở trên)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

rng = np.random.default_rng(0)
x = np.sort(rng.uniform(0, 2 * np.pi, 80)).reshape(-1, 1)
y = np.sin(x).ravel() + rng.normal(0, 0.25, 80)

xs = np.linspace(-1, 2 * np.pi + 1, 400).reshape(-1, 1)   # cố tình vượt ra ngoài miền train

plt.figure(figsize=(9, 4))
plt.scatter(x, y, s=20, color='k', alpha=.6, label='train')
for k, w in [(1, 'uniform'), (5, 'uniform'), (20, 'uniform'), (5, 'distance')]:
    m = KNeighborsRegressor(n_neighbors=k, weights=w).fit(x, y)
    rmse = np.sqrt(mean_squared_error(y, m.predict(x)))
    plt.plot(xs, m.predict(xs), lw=1.8,
             label=f'k={k}, {w} (train RMSE={rmse:.3f}, R2={r2_score(y, m.predict(x)):.3f})')
plt.axvspan(-1, 0, color='gray', alpha=.15)
plt.axvspan(2 * np.pi, 2 * np.pi + 1, color='gray', alpha=.15)
plt.title('KNN cho hồi quy: không ngoại suy ngoài miền train')
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.tight_layout(); plt.show()


Trong hình, $k=1$ và `weights='distance'` đều cho train RMSE bằng 0 và $R^2$ bằng 1: đường dự đoán đi qua đúng từng điểm train, nên hai con số này không dùng để đánh giá được. Ngoài miền train (hai dải xám), mọi đường đều nằm ngang.


## 12. "Lazy learner" và "non-parametric": ý nghĩa thực tế

Phần 2 đã nói KNN là *lazy* và *non-parametric*. Ở đây ta bóc tách xem hai từ đó có nghĩa gì trong thực tế kỹ thuật.

### 12.1. Lazy learner: mô hình chính là dữ liệu

`fit()` của KNN chỉ làm đúng một việc: sao chép tập train vào bộ nhớ (và có thể dựng thêm cây chỉ mục). Không có tham số nào được học, không có gì được nén lại.

| | Eager learner (Linear/Logistic, DT, NN) | Lazy learner (KNN) |
|---|---|---|
| Thời gian train | Chậm (tối ưu hoá) | Gần như bằng 0 |
| Thời gian predict | Rất nhanh (vài phép nhân) | Chậm, phải quét tập train |
| Kích thước model khi deploy | Vài KB (chỉ hệ số) | Bằng cả tập train: $O(Nd)$ |
| Thêm dữ liệu mới | Phải train lại | Chỉ cần `append`, học tăng dần miễn phí |
| Rủi ro riêng tư | Thấp | Cao, file model chứa nguyên dữ liệu gốc |

Ba hệ quả hay bị bỏ quên khi đưa KNN lên production:

1. Bộ nhớ: 1 triệu mẫu × 300 feature × 8 byte $\approx$ 2.4 GB nằm thường trực trong RAM của service.
2. Độ trễ: mỗi request phải tính khoảng cách với toàn bộ tập train (xem mục 13 về cách tăng tốc).
3. Quyền riêng tư / GDPR: xoá dữ liệu của một người dùng khỏi mô hình KNN nghĩa là xoá thật khỏi file model. Không thể "quên" bằng cách train lại như mô hình tham số.

### 12.2. Non-parametric: số "tham số" tăng theo dữ liệu

Non-parametric không có nghĩa là "không có tham số". Nó có nghĩa là mô hình không cố định trước dạng hàm $f$, và độ phức tạp của nó tăng theo $N$:

- Hồi quy tuyến tính $d$ chiều luôn có đúng $d+1$ tham số, dù $N = 100$ hay $N = 10^9$, nên là parametric.
- KNN "nhớ" cả $N$ điểm; càng nhiều dữ liệu, ranh giới càng chi tiết được, nên là non-parametric.

Ưu điểm: không cần giả định phân phối, học được ranh giới hình dạng bất kỳ (kể cả hình xoắn ốc, hình vành khăn). Nhược điểm: cần rất nhiều dữ liệu để lấp đầy không gian, và số dữ liệu cần tăng theo hàm mũ của số chiều, chính là lời nguyền ở mục 14.


## 13. Độ phức tạp tính toán và cách tăng tốc

### 13.1. Brute force

Với mỗi truy vấn, tính khoảng cách tới cả $N$ điểm, mỗi khoảng cách tốn $O(d)$:

$$
\text{1 truy vấn: } O(N d) \qquad \text{tìm top-}k\text{: } O(N \log k) \text{ (dùng heap)} \qquad
M \text{ truy vấn: } O(MNd)
$$

Đây là thứ mà `algorithm='brute'` làm, nhưng nó vectơ hoá được hoàn toàn bằng BLAS (nhờ khai triển $\|u-v\|^2 = \|u\|^2 - 2u\cdot v + \|v\|^2$), nên với $d$ lớn nó thường nhanh hơn cả cây.

### 13.2. KD-Tree

Cây nhị phân chia không gian bằng các siêu phẳng song song trục: mỗi mức chọn một chiều và cắt tại trung vị.

- Dựng cây: $O(dN \log N)$, tốn $O(N)$ bộ nhớ thêm.
- Truy vấn: $\approx O(\log N)$ khi $d$ nhỏ (thực tế $d \lesssim 20$).
- Khi $d$ lớn, phép "cắt tỉa" nhánh gần như không loại được nhánh nào, nên suy biến về $O(N)$ và còn chậm hơn brute force vì mất thêm chi phí duyệt cây.

### 13.3. Ball-Tree

Chia không gian bằng các hình cầu lồng nhau thay vì hộp song song trục. Ưu điểm so với KD-Tree:

- Chịu được $d$ cao hơn (vì hình cầu bám dữ liệu chặt hơn hộp).
- Dùng được với mọi metric thoả bất đẳng thức tam giác (Manhattan, Haversine, Mahalanobis...), không chỉ metric song song trục.

### 13.4. Tham số trong sklearn

| Tham số | Ý nghĩa | Gợi ý |
|---|---|---|
| `algorithm='auto'` | sklearn tự chọn dựa trên $N$, $d$, metric, độ thưa | Cứ để `auto` trước, chỉ ép tay khi đo được lợi ích |
| `'brute'` | Quét toàn bộ | $d$ lớn (> 20-30), dữ liệu thưa (sparse), metric cosine |
| `'kd_tree'` | Cây cắt theo trục | $d$ nhỏ, $N$ lớn, metric Minkowski |
| `'ball_tree'` | Cây hình cầu | $d$ trung bình, metric đặc biệt |
| `leaf_size=30` | Số điểm tối đa ở lá; dưới ngưỡng này cây chuyển sang brute | Tăng lên (50 đến 100) nếu $N$ rất lớn để dựng cây nhanh hơn, ít bộ nhớ hơn; giảm nếu truy vấn nhiều mà dữ liệu ít |
| `n_jobs=-1` | Song song hoá truy vấn | Gần như luôn nên bật khi predict theo lô |

Lưu ý: dữ liệu sparse (ma trận thưa từ `CountVectorizer`, `TfidfVectorizer`) chỉ chạy được với `algorithm='brute'`; cây sẽ báo lỗi. Đó là lý do KNN cho văn bản luôn dùng brute + cosine.

### 13.5. Khi $N$ quá lớn: chuyển sang xấp xỉ (ANN)

Từ khoảng vài triệu điểm trở lên, người ta bỏ tìm kiếm chính xác và dùng **Approximate Nearest Neighbor**: LSH, HNSW (thư viện `faiss`, `hnswlib`, `annoy`). Đánh đổi vài phần trăm recall để lấy tốc độ nhanh hơn hàng trăm lần. Đây chính là công nghệ chạy bên dưới các *vector database* cho RAG/embedding hiện nay. Nói cách khác, ý tưởng KNN vẫn được dùng rộng rãi, chỉ dưới tên gọi khác.


In [ ]:
# Demo độc lập: brute force vs KD-Tree vs Ball-Tree khi số chiều tăng
import time
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)
N, M = 20000, 2000            # 20k điểm train, 2k truy vấn

print(f'{"d":>5} | {"brute (ms)":>11} | {"kd_tree (ms)":>13} | {"ball_tree (ms)":>15}')
print('-' * 54)
for d in [2, 5, 10, 30, 100]:
    X = rng.random((N, d)); y = rng.integers(0, 2, N)
    Q = rng.random((M, d))
    row = []
    for algo in ['brute', 'kd_tree', 'ball_tree']:
        m = KNeighborsClassifier(n_neighbors=5, algorithm=algo).fit(X, y)
        m.predict(Q[:10])                     # làm nóng
        t0 = time.perf_counter(); m.predict(Q); row.append((time.perf_counter() - t0) * 1000)
    print(f'{d:>5} | {row[0]:>11.1f} | {row[1]:>13.1f} | {row[2]:>15.1f}')


Với $d$ nhỏ, hai loại cây nhanh hơn brute force rõ rệt; $d$ càng lớn cây càng suy biến, và đến một ngưỡng thì brute force (vectơ hoá bằng BLAS) lại nhanh nhất. Con số cụ thể phụ thuộc vào máy chạy.


## 14. Lời nguyền chiều cao (curse of dimensionality)

Đây là điểm yếu lớn nhất của KNN, và là vấn đề toán học chứ không phải vấn đề cài đặt.

![Ba biểu hiện của lời nguyền chiều cao](images/06_loi_nguyen_chieu_cao.png)

*Trái: tỷ số $d_{\min}/d_{\max}$ tiến về 1: điểm gần nhất và điểm xa nhất gần như cách đều. Giữa: độ tương phản $(d_{\max}-d_{\min})/d_{\min}$ giảm rất nhanh theo số chiều. Phải: chỉ cần thêm các chiều nhiễu (thông tin hữu ích không đổi), accuracy của KNN giảm từ 98% xuống gần mức ngẫu nhiên.*

### 14.1. Hiện tượng "cô đặc khoảng cách"

Với dữ liệu ngẫu nhiên trong $[0,1]^d$, người ta chứng minh được:

$$
\lim_{d \to \infty} \frac{d_{\max} - d_{\min}}{d_{\min}} = 0
$$

Nghĩa là mọi điểm đều cách nhau xấp xỉ bằng nhau. Mà KNN chỉ có một cơ chế duy nhất: xếp hạng theo khoảng cách. Khi mọi khoảng cách gần bằng nhau, thứ hạng đó chỉ còn là nhiễu, và "hàng xóm gần nhất" không còn nhiều ý nghĩa.

### 14.2. Lập luận thể tích: vì sao "lân cận" không còn "cục bộ"

Muốn một hình hộp con trong $[0,1]^d$ chứa được tỷ lệ $r$ số điểm, cạnh của nó phải dài:

$$
\ell(r) = r^{1/d}
$$

Thử $r = 1\%$ dữ liệu:

| $d$ | $\ell(0.01) = 0.01^{1/d}$ | Diễn giải |
|---|---|---|
| 1 | 0.010 | lân cận thật sự cục bộ |
| 2 | 0.100 | vẫn ổn |
| 10 | 0.631 | phải trải 63% mỗi trục |
| 50 | 0.912 | 91% mỗi trục, "lân cận" = gần như toàn bộ không gian |
| 100 | 0.955 | gần như toàn bộ không gian |

Nói cách khác: ở chiều cao, để gom đủ $k$ hàng xóm bạn buộc phải với ra rất xa, mà ở xa thì $f(x_{(i)})$ không còn liên quan nhiều đến $f(x_0)$, nên bias tăng mạnh.

### 14.3. Cách sống chung

- Chọn feature (bỏ chiều nhiễu): hiệu quả nhất, vì hình trên cho thấy chiều nhiễu là nguyên nhân chính.
- Giảm chiều: PCA, hoặc UMAP/t-SNE cho trực quan hoá (cẩn thận: t-SNE không dùng để giảm chiều cho model).
- Đổi metric: với văn bản/embedding thì cosine chịu chiều cao tốt hơn Euclidean nhiều (vì nó chuẩn hoá độ dài).
- Metric learning (`NeighborhoodComponentsAnalysis` trong sklearn): học một phép biến đổi tuyến tính $L$ rồi dùng $\|L(u-v)\|$, tương đương học một ma trận Mahalanobis biết feature nào đáng tin.
- Đổi mô hình: ở $d$ rất cao và dữ liệu thưa, Logistic Regression / Linear SVM / Naive Bayes / Gradient Boosting hầu như luôn thắng KNN.


## 15. Dữ liệu hỗn hợp (số + phân loại): khoảng cách Gower

Bài thực hành 2 (drug200) đã one-hot rồi tính Euclidean. Cách đó chạy được, nhưng có mấy điểm tế nhị mà sinh viên cần biết trước khi mang lên bài thật.

### 15.1. Vấn đề của "one-hot rồi Euclidean"

1. Khoảng cách giữa hai giá trị phân loại khác nhau luôn là $\sqrt{2}$, bất kể chúng khác nhau nhiều hay ít. `BP = LOW` và `BP = HIGH` cách nhau đúng bằng `BP = LOW` và `BP = NORMAL`, trong khi thực tế BP là biến có thứ tự. Với biến có thứ tự, ordinal encoding phản ánh đúng hơn (đó chính là lý do lab Decision Tree dùng ordinal cho `BP`).
2. Biến phân loại nhiều mức bị nhân trọng số. Một biến 10 mức nở ra 10 cột; tổng đóng góp của nó vào khoảng cách bình phương lớn hơn hẳn một biến 2 mức. Kết quả: KNN vô tình coi biến nhiều mức là quan trọng hơn.
3. `StandardScaler` trên cột one-hot có hai mặt. Một mức hiếm (tần suất $p$ nhỏ) sau khi chuẩn hoá có giá trị $\approx \sqrt{(1-p)/p}$, một con số rất lớn. Nghĩa là *hiếm* bỗng thành *xa*, và mọi mẫu mang mức hiếm đó bị đẩy ra rìa không gian. Đôi khi đó là điều bạn muốn, nhưng phải là một lựa chọn có ý thức. Nếu không muốn, dùng `MinMaxScaler` cho cột số và giữ nguyên cột one-hot ở thang 0/1.

### 15.2. Khoảng cách Gower

Gower (1971) thiết kế riêng cho bảng dữ liệu hỗn hợp. Độ tương đồng giữa hai bản ghi $i, j$:

$$
s(i,j) = \frac{\sum_{f=1}^{p} w_f \, s_f(i,j)}{\sum_{f=1}^{p} w_f}, \qquad d_{\text{Gower}} = 1 - s
$$

trong đó với từng feature $f$:

$$
s_f(i,j) =
\begin{cases}
1 - \dfrac{|x_{if} - x_{jf}|}{R_f} & \text{nếu } f \text{ là số } (R_f = \max_f - \min_f) \\[8pt]
\mathbb{1}[x_{if} = x_{jf}] & \text{nếu } f \text{ là phân loại (nominal)} \\[4pt]
1 - \dfrac{|r_{if} - r_{jf}|}{\text{max rank} - 1} & \text{nếu } f \text{ có thứ tự (dùng hạng } r)
\end{cases}
$$

Ba tính chất khiến Gower rất hợp với bảng dữ liệu thật:

- Mỗi feature đóng góp đúng một phiếu trong $[0,1]$, bất kể kiểu dữ liệu hay số mức, nên không còn chuyện biến nhiều mức lấn át.
- Chuẩn hoá đã nằm sẵn trong công thức (chia cho $R_f$), nên không cần scaler riêng.
- Trọng số $w_f$ cho phép nhấn mạnh feature quan trọng hoặc bỏ qua ô thiếu dữ liệu ($w_f = 0$).

sklearn không có Gower sẵn. Cách dùng: tự tính ma trận khoảng cách rồi truyền vào

```python
knn = KNeighborsClassifier(n_neighbors=5, metric='precomputed')
knn.fit(D_train_train, y_train)      # ma trận N_train x N_train
knn.predict(D_test_train)            # ma trận N_test x N_train
```

(hoặc dùng gói ngoài như `gower`). Với bài lab thì one-hot + scale vẫn ổn, chỉ cần bạn biết mình đang đánh đổi cái gì.


## 16. Dữ liệu mất cân bằng làm hỏng biểu quyết KNN như thế nào

Giả sử lớp A chiếm 95% và lớp B chiếm 5%. Ngay cả tại một vị trí $x_0$ mà mật độ hai lớp bằng nhau, số hàng xóm kỳ vọng thuộc mỗi lớp vẫn tỷ lệ với **prior**:

$$
\mathbb{E}[\text{votes}_c] \;\approx\; k \cdot \frac{\pi_c\, p(x_0 \mid c)}{\sum_{c'} \pi_{c'}\, p(x_0 \mid c')}
$$

Với $\pi_A = 0.95$, lớp B phải có mật độ gấp 19 lần lớp A tại $x_0$ thì mới hoà phiếu. Hệ quả rất cụ thể:

- $k$ càng lớn, lớp thiểu số càng khó thắng phiếu. Với $k=50$ và tỷ lệ 95:5, cần ít nhất 26 trong 50 hàng xóm là lớp B, gần như không xảy ra ngoài vùng lõi của lớp B.
- Ranh giới bị đẩy lấn vào vùng của lớp thiểu số, nên recall của lớp hiếm rất thấp dù accuracy tổng thể trông cao.
- Đây là lý do accuracy không phải chỉ số phù hợp với dữ liệu mất cân bằng: mô hình cơ sở luôn đoán lớp đa số đã đạt 95%.

### Cách xử lý (theo thứ tự nên thử)

| Cách | Làm gì | Ghi chú |
|---|---|---|
| Đổi thước đo | Dùng `f1_score`, `balanced_accuracy_score`, PR-AUC, `classification_report` | Bắt buộc, làm trước mọi thứ khác |
| Giảm $k$ | $k$ nhỏ (3 đến 5) giữ được túi nhỏ của lớp hiếm | Đánh đổi bằng variance cao hơn |
| `weights='distance'` | Hàng xóm hiếm nhưng rất gần vẫn có tiếng nói | Giúp một phần, không giải quyết triệt để |
| Chỉnh ngưỡng | Dùng `predict_proba` rồi hạ ngưỡng cho lớp hiếm | Rất hiệu quả, không đụng đến dữ liệu |
| Resampling | `RandomUnderSampler`, `SMOTE` (gói `imbalanced-learn`) | Chỉ resample trên tập train, và phải nằm trong `Pipeline` để không leakage vào CV |
| Đổi mô hình | Cây/RF có `class_weight='balanced'`, KNN thì không có | Nếu mất cân bằng quá nặng, cân nhắc bỏ KNN |

Một lưu ý về SMOTE với KNN: SMOTE sinh mẫu tổng hợp bằng cách nội suy giữa các hàng xóm, nghĩa là nó dùng chính KNN. Nếu chạy SMOTE trên toàn bộ dữ liệu trước khi split, mẫu tổng hợp sẽ mang thông tin từ tập test, gây leakage và điểm số cao hơn thực tế.


## 17. Khi nào dùng KNN, khi nào tránh

| | Nên dùng KNN | Nên tránh KNN |
|---|---|---|
| Số chiều | $d$ nhỏ đến trung bình (< 20-30) sau khi đã chọn feature | $d$ rất lớn, đặc biệt là nhiều chiều nhiễu |
| Số mẫu | $N$ vừa phải (vài nghìn đến vài trăm nghìn) | $N$ hàng triệu mà cần predict thời gian thực |
| Hình dạng ranh giới | Phức tạp, phi tuyến, nhiều cụm rời rạc | Tuyến tính rõ ràng (Logistic/SVM tuyến tính rẻ hơn nhiều) |
| Yêu cầu tốc độ | Train phải nhanh / dữ liệu cập nhật liên tục | Latency predict phải cực thấp, thiết bị ít RAM |
| Yêu cầu giải thích | "Vì 5 ca giống nhất đều là X", rất thuyết phục với người dùng cuối | Cần công thức, hệ số, hay quy tắc if-then in ra được |
| Dữ liệu | Đầy đủ, sạch, đã chuẩn hoá, ít mất cân bằng | Nhiều giá trị thiếu, mất cân bằng nặng, nhiều biến phân loại nhiều mức |
| Vai trò | Baseline nên thử trước khi đụng mô hình phức tạp | Mô hình cuối cùng cho hệ thống lớn |

Những ứng dụng thực tế của KNN:

- Hệ gợi ý kiểu *item-based / user-based collaborative filtering*: bản chất là KNN trên ma trận tương tác.
- Tìm kiếm ngữ nghĩa và RAG: vector database (FAISS, Milvus, pgvector) chính là KNN xấp xỉ trên embedding.
- Nhận dạng khuôn mặt / chống trùng lặp: so embedding với cơ sở dữ liệu đã đăng ký.
- Phát hiện bất thường: khoảng cách tới hàng xóm thứ $k$ chính là điểm bất thường (thuật toán LOF là một biến thể của KNN).
- Điền giá trị thiếu: `sklearn.impute.KNNImputer`.


## 18. Danh sách bẫy: kiểm tra trước khi nộp bài

1. **Quên chuẩn hoá.** KNN không chuẩn hoá thì gần như luôn sai. Kiểm tra `X_train.std(axis=0)`: nếu các con số lệch nhau vài bậc thì bạn đang có vấn đề.
2. **`fit` scaler trên cả tập (leakage).** Đúng: `scaler.fit(X_train)` rồi `scaler.transform(X_test)`. An toàn nhất: bọc trong `Pipeline` rồi mới `cross_val_score`.
3. **Chọn $k$ bằng tập test.** Đó là leakage trá hình: điểm test bạn báo cáo không còn khách quan. Chọn $k$ bằng CV trên train, chỉ chạm vào test đúng một lần ở cuối.
4. **$k$ chẵn với bài 2 lớp** dẫn tới hoà phiếu, và sklearn lặng lẽ thiên vị lớp có nhãn nhỏ hơn. Nên dùng $k$ lẻ.
5. **Báo cáo train accuracy của $k=1$ (hoặc của `weights='distance'`).** Luôn bằng 100% theo định nghĩa, một con số không có giá trị đánh giá.
6. **Nhầm cosine similarity với cosine distance.** `distance = 1 − similarity`. Sắp xếp tăng dần cho distance, giảm dần cho similarity. Và nhớ: nếu dùng cosine thì nên chuẩn hoá L2 từng vector trước (`sklearn.preprocessing.normalize`), lúc đó Euclidean và cosine cho cùng thứ hạng vì $\|u-v\|^2 = 2(1 - \cos(u,v))$ khi $\|u\|=\|v\|=1$.
7. **Dùng KNN trên dữ liệu thưa cao chiều mà quên `algorithm='brute'`.** KD-Tree/Ball-Tree không nhận ma trận sparse; và ở $d$ hàng nghìn thì cây cũng vô dụng.
8. **Để `n_jobs` mặc định khi predict theo lô lớn.** `n_jobs=-1` thường nhanh hơn nhiều lần mà không tốn công gì.
9. **Áp dụng KNN cho dữ liệu mất cân bằng rồi báo cáo accuracy.** Xem lại mục 16.
10. **Quên rằng model KNN chứa nguyên dữ liệu train.** Đừng gửi file `.pkl` của KNN cho bên thứ ba nếu dữ liệu là dữ liệu nhạy cảm.


## Tổng kết

1. KNN dự đoán bằng cách bầu chọn từ $k$ hàng xóm gần nhất.
2. Phải scale feature trước khi dùng (vì KNN dùng khoảng cách).
3. Phải fit scaler chỉ trên train, rồi transform test để tránh data leakage.
4. Chọn $k$ bằng cross-validation, không phải tuỳ ý.
5. Phân biệt cosine similarity (lớn là gần) và cosine distance = 1 − similarity (nhỏ là gần).
6. Nearest Centroid là họ hàng đơn giản, nhanh hơn nhưng kém linh hoạt.

# BÀI TẬP VỀ NHÀ

## Bài 1: So sánh metric khoảng cách
Trên Iris, train `KNeighborsClassifier(n_neighbors=5)` với 3 metric: `euclidean`, `manhattan`, `chebyshev`. So sánh test accuracy.

*Gợi ý:* `KNeighborsClassifier(n_neighbors=5, metric='manhattan')`.

## Bài 2: Trọng số khoảng cách
Mặc định KNN đếm hàng xóm bằng phiếu bằng nhau. Có thể đổi sang trọng số `weights='distance'`: hàng xóm gần hơn có quyền hơn. Trên drug200, so sánh `weights='uniform'` vs `weights='distance'` với k = 5, 11, 15. Cái nào tốt hơn? Vì sao?

## Bài 3: Vẽ decision boundary
Lấy 2 feature đầu của Iris (sepal length, sepal width). Train KNN với k = 1, 5, 30. Vẽ decision boundary cho mỗi k bằng `plt.contourf` trên lưới điểm. Quan sát: k nhỏ cho boundary lởm chởm (overfit), k lớn cho boundary mượt nhưng có thể bỏ chi tiết.

*Gợi ý:* tạo lưới `np.meshgrid`, predict lưới, vẽ contour.

## Bài 4: KNN cho text
Trên dataset `Education.csv` (đã dùng ở lab Naive Bayes), thử dùng KNN. Yêu cầu:
1. `CountVectorizer` để vector hoá văn bản.
2. KNN với `metric='cosine'`.
3. Sweep $k \in \{1, 3, 5, 7\}$.
4. So sánh với Multinomial NB (đã có ở bài Naive Bayes). Kết luận?

## Bài 5: Curse of dimensionality
Sinh dữ liệu giả 2 lớp: 100 mẫu, $d$ chiều Gaussian, trong đó chỉ 2 chiều đầu có tâm hai lớp lệch nhau (ví dụ lệch 2 đơn vị), các chiều còn lại là nhiễu thuần với cùng phân phối ở cả hai lớp. Cho $d \in \{2, 10, 50, 200\}$, đo accuracy của KNN(k=5) trên tập test. Quan sát: khi $d$ tăng, KNN kém dần. Đây là lời nguyền chiều cao: ở chiều cao, mọi điểm cách nhau gần như bằng nhau, nên khái niệm "hàng xóm gần" không còn ý nghĩa.